# Loading the compact L2 table
Original exploratory loading example. The report figures are generated by the numbered scripts.
Place input data in `data/bi_aggregates_compact`, or set `CS5488_DATA_DIR` before opening the notebook.

In [ ]:
from pathlib import Path
import pandas as pd

# 1. 指向 L2 文件夹，不用逐个打开 731 个文件
import os
package_root = Path.cwd() if (Path.cwd() / "run_analysis.py").exists() else Path.cwd().parent
data_root = Path(os.environ.get("CS5488_DATA_DIR", str(package_root / "data/bi_aggregates_compact"))).expanduser()
folder = data_root / "l2_overview"

# 2. 只读取画图需要的四列
l2 = pd.read_parquet(
    folder,
    engine="pyarrow",
    columns=["pickup_date", "pickup_hour", "PULocationID", "trip_count"],
    partitioning=None,
)

# 3. 检查：行数是聚合组数，trip_count 加总才是行程数
print(l2.head())
print(f"聚合行数：{len(l2):,}")
print(f"行程总数：{l2['trip_count'].sum():,}")

# 4. 同一天有很多“小时 × 区域”组，把它们的行程数加起来
daily = l2.groupby("pickup_date", observed=True)["trip_count"].sum()
daily.index = pd.to_datetime(daily.index.astype(str))
daily = daily.sort_index()

print(daily.head())